# 04 — QAOA depth sweep

**Sweep 1.** Fixed problem (`n=8, K=2, mag7`-like universe). Vary `p in {1..5}`.
For each `p` we run 10 multi-start COBYLA seeds and keep the best, then report
approximation ratio, P(optimum), and P(feasible) across `p`.

Results cache to `results/depth_sweep.json` so re-plotting is instant.

In [ ]:
# === Bootstrap (Colab + local) ===
import sys, os, json
try:
    import google.colab  # noqa: F401
    get_ipython().system('test -d /content/fys5419 || git clone -q https://github.com/egil10/fys5419.git /content/fys5419')
    get_ipython().run_line_magic('cd', '/content/fys5419/project2/code/notebooks')
except ImportError:
    pass
sys.path.append('..')
from scripts.colab import setup; setup()

# === Project imports ===
from pathlib import Path
import numpy as np
import pandas as pd

from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import brute_force
from scripts.qaoa      import solve
from scripts.metrics   import prob_optimal, prob_feasible

RESULTS = Path.cwd().parent / 'results'
RESULTS.mkdir(exist_ok=True)

In [ ]:
# === Canonical instance: n=16, K=4 ===
P_VALUES   = [1, 2, 3, 4, 5]
N_RESTARTS = 10

r  = load_universe()
pf = PortfolioProblem(r.mu, r.Sigma,
                      lam=DEFAULTS['lam'], A=DEFAULTS['A'],
                      K=DEFAULTS['K_AT_16'], tickers=r.tickers)
bf = brute_force(pf)
print(f'brute force: {bf.bitstring}  C={bf.cost:.6f}  picks={bf.tickers(pf)}')

In [ ]:
cache = RESULTS / 'depth_sweep.json'

if cache.exists():
    sweep = json.loads(cache.read_text())
    print(f'loaded {cache.name}')
else:
    sweep = []
    for p in P_VALUES:
        res = solve(pf, p=p, n_restarts=N_RESTARTS, seed=42)
        sweep.append({
            'p':           p,
            'energy':      float(res['energy']),
            'ratio':       float(res['ratio']),
            'p_optimal':   prob_optimal(res['probs'], bf.x),
            'p_feasible':  prob_feasible(res['probs'], pf.n, pf.K),
            'best_C':      float(min(h['fun'] for h in res['history'])),
        })
        print(f'  p={p}: ratio={res["ratio"]:.4f}  '
              f'P(opt)={sweep[-1]["p_optimal"]:.4f}  '
              f'P(feas)={sweep[-1]["p_feasible"]:.4f}')
    cache.write_text(json.dumps(sweep, indent=2))
    print(f'saved -> {cache.name}')

pd.DataFrame(sweep)